In [1]:
#Libraries
import pandas as pd
import numpy as np
import re
from transformers import AutoTokenizer
from datasets import Dataset
import torch
import torch.nn as nn
from transformers import AutoModel
from transformers import DataCollatorWithPadding
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [6]:
#Load Dataset
train_df = pd.read_csv("/content/phishing_train.csv")
val_df = pd.read_csv("/content/phishing_validation.csv")
test_df = pd.read_csv("/content/phishing_test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(14015, 4)
(1752, 4)
(1752, 4)


In [7]:
#Create feature extractor
URGENCY_WORDS = {
    "urgent", "urgently", "immediately", "immediate",
    "now", "today", "asap", "quickly", "deadline",
    "expire", "expired", "final", "action"
}

CREDENTIAL_WORDS = {
    "password", "passwd", "username", "login",
    "credential", "credentials", "verify", "verification",
    "authenticate", "authentication", "account"
}

THREAT_WORDS = {
    "suspend", "suspended", "terminate", "terminated",
    "blocked", "block", "close", "closed", "penalty",
    "fraud", "unauthorized", "warning", "security"
}

FINANCIAL_WORDS = {
    "payment", "pay", "invoice", "money", "bank",
    "transfer", "transaction", "refund", "credit",
    "debit", "fee", "account", "billing"
}

CTA_WORDS = {
    "click", "clicking", "visit", "open", "download",
    "confirm", "verify", "submit", "update",
    "activate", "login"
}


def count_terms(text, vocabulary):
    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    return sum(word in vocabulary for word in words)


def extract_nlp_features(text):

    text = str(text)

    words = re.findall(r"\b[a-zA-Z]+\b", text)

    word_count = max(len(words), 1)

    uppercase_words = [
        word for word in words
        if len(word) > 1 and word.isupper()
    ]

    uppercase_ratio = len(uppercase_words) / word_count

    url_count = len(
        re.findall(
            r"https?://\S+|www\.\S+",
            text,
            flags=re.IGNORECASE
        )
    )

    email_count = len(
        re.findall(
            r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
            text
        )
    )

    phone_count = len(
        re.findall(
            r"(?:\+?\d[\d\s().-]{7,}\d)",
            text
        )
    )

    features = [
        count_terms(text, URGENCY_WORDS),
        count_terms(text, CREDENTIAL_WORDS),
        count_terms(text, THREAT_WORDS),
        count_terms(text, FINANCIAL_WORDS),
        count_terms(text, CTA_WORDS),
        url_count,
        email_count,
        phone_count,
        text.count("!"),
        text.count("?"),
        uppercase_ratio,
        np.log1p(len(text))
    ]

    return np.array(features, dtype=np.float32)

In [8]:
#Generate features
train_features = np.vstack(
    train_df["processed_text"].apply(extract_nlp_features)
)

val_features = np.vstack(
    val_df["processed_text"].apply(extract_nlp_features)
)

test_features = np.vstack(
    test_df["processed_text"].apply(extract_nlp_features)
)

print("Feature shape:", train_features.shape)
print("First feature vector:")
print(train_features[0])

Feature shape: (14015, 12)
First feature vector:
[0.       0.       1.       0.       0.       0.       0.       0.
 0.       1.       0.       6.368187]


In [9]:
#Feature normalisation
feature_mean = train_features.mean(axis=0)
feature_std = train_features.std(axis=0)

feature_std[feature_std == 0] = 1.0

train_features = (
    (train_features - feature_mean) / feature_std
)

val_features = (
    (val_features - feature_mean) / feature_std
)

test_features = (
    (test_features - feature_mean) / feature_std
)

print("Feature normalisation complete.")

Feature normalisation complete.


In [10]:
#DeBERT tokenisation
MODEL_NAME = "microsoft/deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["processed_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

In [11]:
#Convert dataset
train_dataset = Dataset.from_pandas(
    train_df[["processed_text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["processed_text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["processed_text", "label"]],
    preserve_index=False
)

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)
val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)

Map:   0%|          | 0/14015 [00:00<?, ? examples/s]

Map:   0%|          | 0/1752 [00:00<?, ? examples/s]

Map:   0%|          | 0/1752 [00:00<?, ? examples/s]

In [12]:
#Add explicit NLP feature
train_tokenized = train_tokenized.add_column(
    "nlp_features",
    train_features.tolist()
)

val_tokenized = val_tokenized.add_column(
    "nlp_features",
    val_features.tolist()
)

test_tokenized = test_tokenized.add_column(
    "nlp_features",
    test_features.tolist()
)

print(train_tokenized)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask', 'nlp_features'],
    num_rows: 14015
})


In [13]:
#Build the model
class ModelDraft1(nn.Module):

    def __init__(self, model_name, num_features=12, num_labels=2):
        super().__init__()

        # Pretrained transformer backbone
        self.transformer = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32
        )

        hidden_size = self.transformer.config.hidden_size

        # Explicit NLP feature branch
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(hidden_size + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Final classifier
        self.classifier = nn.Linear(256, num_labels)

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        nlp_features=None,
        labels=None
    ):

        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # DeBERTa contextual representation
        contextual_embedding = outputs.last_hidden_state[:, 0, :]

        # Explicit NLP representation
        feature_embedding = self.feature_projection(
            nlp_features.float()
        )

        # Feature fusion
        combined = torch.cat(
            [contextual_embedding, feature_embedding],
            dim=1
        )

        fused = self.fusion(combined)

        logits = self.classifier(fused)

        loss = None

        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

In [14]:
#Load the model
model = ModelDraft1(
    MODEL_NAME,
    num_features=12,
    num_labels=2
)

print(
    "Model dtype:",
    next(model.parameters()).dtype
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model dtype: torch.float32


In [16]:
#Set data collator
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [19]:
#Evaluating matrix
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [23]:
#Setup configure
training_args = TrainingArguments(
    output_dir="/content/modeldraft1_results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    gradient_accumulation_steps=2,

    learning_rate=1e-5,
    weight_decay=0.01,

    warmup_steps=263,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=False,
    bf16=False,

    optim="adamw_torch",

    seed=42,
    data_seed=42,

    report_to="none",
    save_total_limit=2
)

In [3]:
#Trainer
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Model Draft1 Trainer ready.")

NameError: name 'model' is not defined

In [25]:
#Train the model
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.044914,0.041841,0.990868,0.993808,0.981651,0.987692
2,0.017469,0.046758,0.992009,0.990798,0.987768,0.989280
3,0.004125,0.052524,0.992580,0.993837,0.986239,0.990023


In [26]:
#Evaluate on test set
test_results = trainer.evaluate(
    test_tokenized,
    metric_key_prefix="test"
)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.004125,0.035094,3,0.993721,0.995378,0.987768,0.991558


{'test_loss': 0.0350944846868515, 'test_accuracy': 0.9937214611872146, 'test_precision': 0.9953775038520801, 'test_recall': 0.9877675840978594, 'test_f1': 0.9915579432079816}
